# PROJECT STAGE

Current stage: Research exploration

Code may later be migrated to src modules once stabilized.

# 1. Notebook Metadata

In [ ]:
# ============================================================
# NOTEBOOK: 01_data_pipeline.ipynb
#
# Quant Research Platform
#
# PURPOSE
# -------
# Build the canonical market dataset used by the research system.
#
# PIPELINE
# --------
# Raw Market Data
#     ↓
# Price Cleaning (crypto + equities)
#     ↓
# Calendar Alignment (union de fechas)
#     ↓
# Return Computation
#     ↓
# Processed Datasets
#
# OUTPUT DATASETS
# ---------------
# data/processed/prices.parquet
# data/processed/returns.parquet
#
# NOTES
# -----
# - Data source: Yahoo Finance (yfinance)
# - Returns: log returns
# - Maintains crypto fines de semana
# ============================================================

# 2. Imports

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
from pathlib import Path
import pyarrow
import datetime


import plotly.express as px

# 3. Configuration

In [ ]:
# ===============================
# ASSET UNIVERSE
# ===============================

ASSETS = {
    "SPY": "SPY",        # US equities (S&P500)
    "QQQ": "QQQ",        # Nasdaq / technology
    "XLE": "XLE",        # Energy sector
    "BTC": "BTC-USD"     # Bitcoin (global liquidity proxy)
}


# ===============================
# DATA PATHS
# ===============================

DATA_DIR = Path("../data")
PROCESSED_DIR = DATA_DIR / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


# ===============================
# DATA PARAMETERS
# ===============================

START_DATE = "2015-01-01"

# 4. Data Loading

In [ ]:
# ===============================
# DOWNLOAD MARKET DATA
# ===============================

prices = yf.download(
    list(ASSETS.values()),
    start=START_DATE,
    auto_adjust=True,
    progress=False
)["Close"]


# ===============================
# RENAME COLUMNS (SAFE MAPPING)
# ===============================

prices = prices.rename(columns={v: k for k, v in ASSETS.items()})


# Ensure consistent column order
prices = prices[list(ASSETS.keys())]

prices.tail()


# 5. Core Computation

In [ ]:
# ===============================
# PRICE CLEANING & CALENDAR ALIGNMENT
# ===============================

# Ensure chronological order
prices = prices.sort_index()

# Forward-fill limitado para equities (SPY, QQQ, XLE)
equities = ["SPY", "QQQ", "XLE"]
prices[equities] = prices[equities].ffill(limit=5)  # cubre feriados + findes

# BTC mantiene su valor real en fines de semana
# Drop initial NaNs that may appear due to reindexing
prices = prices.dropna(how="any")

# ===============================
# RETURN COMPUTATION
# ===============================

returns = np.log(prices).diff().dropna()

prices.head()


# 6. Diagnostics/validation

In [ ]:
# ===============================
# DATASET DIAGNOSTICS
# ===============================

print("Prices shape:", prices.shape)
print("Returns shape:", returns.shape)

print("\nDate range:")
print(prices.index.min(), "→", prices.index.max())

print("\nMissing values:")
print(prices.isna().sum())

print("\nAssets in dataset:")
print(list(prices.columns))


# ===============================
# PIPELINE CONSISTENCY CHECK
# ===============================

assert set(prices.columns) == set(ASSETS.keys()), "Asset mismatch"

assert prices.index[1:].equals(returns.index), "Return index mismatch"

print("\nPipeline validation successful")

# 7. Visualization

In [ ]:
# ===============================
# PRICE SERIES
# ===============================

fig = px.line(
    prices,
    title="Asset Prices"
)

fig.show()

In [ ]:
fig = px.line(
    returns,
    title="Log Returns"
)

fig.add_hline(y=0, line_dash="dash")

fig.show()

In [ ]:
cumulative_returns = np.exp(returns.cumsum())

fig = px.line(
    cumulative_returns,
    title="Cumulative Performance (Growth of $1)"
)

fig.show()

# 8. Export Results

In [ ]:
# ===============================
# SAVE PROCESSED DATASETS
# ===============================

prices.to_parquet(PROCESSED_DIR / "prices.parquet")

returns.to_parquet(PROCESSED_DIR / "returns.parquet")

print("Datasets exported successfully")